# Residential episodes — Lausanne trajectories

**Pipeline notebook 5 (Phase 3).** Builds the episode table — one residential stay
per row — by chaining consolidated register events per individual, anchored on the
2014 year-end snapshot. This table is the pivot structure feeding both deliverables
(departure model features, trajectory clustering).

**Scope decisions (established with the data owner):**

- **Unit of an episode = the dwelling `(EGID, EWID)`**, with `NUMERO_MENAGE` as a
  mutable attribute (several households can transiently share a dwelling when a
  departure is not yet confirmed).
- **Principal residence only** — rows with `TYPAD_C = Secondaire` are excluded
  (counted).
- **Private and collective households only** — episodes exist only in
  `TYPE_MENAGE ∈ {Prive, Collectif}`. Administrative pseudo-addresses and
  out-of-commune states are not episode locations; *people* passing through them
  are kept, the interval is simply not an episode.
- **Absences do not close episodes.** Out-of-commune moves, hospitalisation and
  detention keep the episode open at the last known dwelling with an absence flag;
  a side table `absences` records each interval (consistent with `CODETA_C`
  staying `Actif` and with the notebook-3 target).
- **Purely spatio-temporal**: civil-status/permit events do not open, close or
  annotate episodes (they stay in `masterfile_events` for feature engineering).

**Operating rules carried forward:**

- **Two time axes**: `DATE_EFFECTIVE` (biographical time) orders the chaining;
  `MUTATION_DATE` (administrative entry time) resolves consolidation — monthly
  files are journals of *entries*, not of the month's events (retroactive depth
  observed back to 1990).
- **Transitions are detected by state comparison, not by event family**: any
  consolidated row where `(MUTATION_EGID, MUTATION_EWID) ≠ (EGID, MENAGE_EWID)`
  is a spatial transition (corrections and household regroupings move people too);
  the family only labels the transition. Note the asymmetric column naming:
  post-state EWID lives in `MENAGE_EWID`, pre-state in `MUTATION_EWID`.
- **Consolidation before chaining** (§2): (a) `Suppression*` matched to their
  target via the crossed-address lock — `supp.MUTATION_* = cancelled state`,
  `supp normal = restored state` — with dates as tie-breaker only (their
  `DATE_EFFECTIVE` semantics is inconsistent: sometimes the cancelled event's
  date, often `DATARR` of the restored stay); (b) last-write-wins deduplication
  of re-entered events; (c) corrections applied as amendments.
- **Arrivals open chains**: their `MUTATION_*` may point to a former Lausanne
  file (returning residents) and is informative, not a continuity constraint.
- Departure-side rules from notebook 3: `DepartNonConfirme` closes immediately,
  `DepartAnticipe` closes at announced `DATDEP` once reached; deaths close with
  `fin_type = deces`; `depart_status ∈ {death, dossier_supprime}` individuals are
  excluded from the modelling population downstream (kept here for reconciliation).
- Episode 0 is seeded from `population_2014` (start = arrival date when carried by
  the snapshot, else 2014-12-31 flagged as an administrative left bound); events
  effective *before* the window are reconciled against the snapshot, not rejected.

## 0. Load pipeline inputs

Three inputs: `masterfile_events` (notebook 3 — audited master + event
classification), `person_target` (notebook 3 — per-individual target and status),
`population_2014` (notebook 4 — normalized year-end snapshot). The date guard
tries ISO 8601 first (pandas' own CSV dialect, hit when the parquet fallback was
used) and falls back to the Swiss registry format only if ISO clearly fails.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("/mnt/o/09_STATISTIQUES/09_04_Explorations/MR/CAS ADS/Final project/data")

DATE_FMT = "%d.%m.%Y %H:%M"

def load(stem):
    pq, gz = DATA_DIR / f"{stem}.parquet", DATA_DIR / f"{stem}.csv.gz"
    if pq.exists():
        return pd.read_parquet(pq)
    if gz.exists():
        return pd.read_csv(gz, dtype=str, low_memory=False)
    raise FileNotFoundError(f"{stem}: run the upstream notebook first.")

def parse_dates(df, cols):
    for c in cols:
        if c in df.columns and not pd.api.types.is_datetime64_any_dtype(df[c]):
            parsed = pd.to_datetime(df[c], format="ISO8601", errors="coerce")
            if parsed.isna().mean() > 0.5:            # not ISO -> Swiss registry format
                parsed = pd.to_datetime(df[c], format=DATE_FMT, errors="coerce")
            df[c] = parsed
    return df

EVENT_DATES = ["DATE_EFFECTIVE", "MUTATION_DATE", "DATDEP", "DDECES", "DATNAIS", "DATARR", "DATDEM"]

events  = parse_dates(load("masterfile_events"), EVENT_DATES)
target  = load("person_target")
pop2014 = parse_dates(load("population_2014"), EVENT_DATES)

# Columns the chaining logic depends on -- fail loudly if a schema drifted.
REQUIRED = ["id_projet", "CODE_MUTATION", "famille_evenement", "nature_evenement",
            "DATE_EFFECTIVE", "MUTATION_DATE", "DATDEP", "DATARR", "DATNAIS", "DDECES",
            "EGID", "MENAGE_EWID", "MUTATION_EGID", "MUTATION_EWID",
            "TYPE_MENAGE", "MUTATION_TYPEMENAGE", "NUMERO_MENAGE", "MUTATION_NUMERO_MENAGE",
            "PERSMEN", "TYPAD_C", "TYPADRES_C", "MUTATION_TYPADRES_C"]
missing = [c for c in REQUIRED if c not in events.columns]
assert not missing, f"masterfile_events is missing required columns: {missing}"

CENSOR_DATE = pd.Period(events["periode"].max(), freq="M").end_time

print(f"masterfile_events : {len(events):,} rows | {events['id_projet'].nunique():,} individuals")
print(f"  DATE_EFFECTIVE  : {events['DATE_EFFECTIVE'].min().date()} -> {events['DATE_EFFECTIVE'].max().date()}"
      f"  (NaT: {events['DATE_EFFECTIVE'].isna().sum():,})")
print(f"  MUTATION_DATE   : {events['MUTATION_DATE'].min().date()} -> {events['MUTATION_DATE'].max().date()}"
      f"  (NaT: {events['MUTATION_DATE'].isna().sum():,})")
print(f"  censor date     : {CENSOR_DATE.date()}")
print(f"person_target     : {len(target):,} individuals | "
      f"departed=1: {pd.to_numeric(target['departed']).sum():,}")
print(f"population_2014   : {len(pop2014):,} rows x {pop2014.shape[1]} columns")

# Snapshot <-> events reconciliation preview (full validation in the final section)
if "id_projet" in pop2014.columns:
    overlap = pop2014["id_projet"].isin(set(events["id_projet"])).mean()
    print(f"  snapshot individuals also seen in events: {overlap*100:.1f}%")
else:
    print("  WARNING: no id_projet column in population_2014 -- check notebook 4 rename map:")
    print("  ", list(pop2014.columns)[:15], "...")

# Encoding sanity check (mojibake would mean notebook 1 misdetected an encoding)
moji = events["LIBADRES_C"].astype(str).str.contains("Ã", na=False).sum()        if "LIBADRES_C" in events.columns else 0
print(f"mojibake check    : {moji:,} rows with suspicious 'Ã' in LIBADRES_C"
      + ("  <- investigate notebook 1 encoding detection!" if moji else " (clean)"))

masterfile_events : 863,577 rows | 291,763 individuals
  DATE_EFFECTIVE  : 1913-03-25 -> 2027-08-01  (NaT: 0)
  MUTATION_DATE   : 2015-01-01 -> 2026-05-31  (NaT: 0)
  censor date     : 2026-05-31
person_target     : 291,763 individuals | departed=1: 166,488
population_2014   : 140,228 rows x 59 columns
  snapshot individuals also seen in events: 71.6%
mojibake check    : 0 rows with suspicious 'Ã' in LIBADRES_C (clean)


## 0bis. Date integrity gate

**History.** The first run of this diagnostic exposed a corpus-wide parsing failure:
across all 136 batches, zero parsed dates had a day > 12 (statistically impossible),
NaT rates (~54–59%) matched the share of month days > 12, and 154,845 entry
timestamps postdated their own file's month with a tell-tale seasonality (every
December at exactly zero). Diagnosis (final, established by the raw-string inventory of notebook 2 §1): the
source strings are ISO year-first (`2015-01-08 16:22:21.0`) — the data was never
malformed. The corruption came from a pandas ≥ 2 trap: with no explicit format,
pandas guesses one from the first element, and `dayfirst=True` makes it guess
`%Y-%d-%m` whenever both middle fields of that element are ≤ 12 — swapping month
and day corpus-wide and coercing true days > 12 to NaT. The fix (notebook 2 §1)
detects the convention empirically per column and parses with explicit formats;
notebook 3 was re-run since terminal states and the target were computed on the
corrupted dates.

**This cell is now the gate**: notebook 5 refuses to chain on dates that fail the
three signatures. Expected after the fix — day>12 share ≈ 0.6 on every batch, no
entry postdating its file month, NaT rates low and structural only.

In [2]:
per_end = pd.PeriodIndex(events["periode"], freq="M").end_time

diag = pd.DataFrame({
    "periode":   events["periode"],
    "de_nat":    events["DATE_EFFECTIVE"].isna(),
    "md_nat":    events["MUTATION_DATE"].isna(),
    "de_day13":  events["DATE_EFFECTIVE"].dt.day > 12,
    "md_day13":  events["MUTATION_DATE"].dt.day > 12,
    "md_future": events["MUTATION_DATE"] > per_end,
})
g = diag.groupby("periode").agg(
    de_nat=("de_nat", "mean"), md_nat=("md_nat", "mean"), md_future=("md_future", "sum"))
g["de_day13"] = diag.loc[~diag["de_nat"]].groupby("periode")["de_day13"].mean()
g["md_day13"] = diag.loc[~diag["md_nat"]].groupby("periode")["md_day13"].mean()

print("Global: DATE_EFFECTIVE NaT {:.1%} | MUTATION_DATE NaT {:.1%} | "
      "day>12 shares {:.2f} / {:.2f} | future entries {:,}".format(
      diag["de_nat"].mean(), diag["md_nat"].mean(),
      diag.loc[~diag["de_nat"], "de_day13"].mean(),
      diag.loc[~diag["md_nat"], "md_day13"].mean(),
      int(diag["md_future"].sum())))

bad = g[(g["de_day13"] < 0.40) | (g["md_day13"] < 0.40) | (g["md_future"] > 0)
        | (g["de_nat"] > 0.20) | (g["md_nat"] > 0.20)]
if len(bad):
    print(f"\n{len(bad)} batch(es) failing the gate:")
    print(bad.to_string(float_format=lambda x: f"{x:.2f}"))

# Hard gate -- do not chain on corrupted dates.
assert diag.loc[~diag["de_nat"], "de_day13"].mean() > 0.40, "DATE_EFFECTIVE: day/month swap signature still present"
assert diag.loc[~diag["md_nat"], "md_day13"].mean() > 0.40, "MUTATION_DATE: day/month swap signature still present"
assert int(diag["md_future"].sum()) == 0, "entry timestamps postdating their file month -- rerun notebooks 2-3"
print("\nGate PASSED -- dates are chainable.")

Global: DATE_EFFECTIVE NaT 0.0% | MUTATION_DATE NaT 0.0% | day>12 shares 0.54 / 0.59 | future entries 0

Gate PASSED -- dates are chainable.


## 1. Scope filters & spatial vocabulary

Applies the agreed perimeter and builds the location vocabulary the chaining will
speak. Every exclusion is counted (report material).

**Scope (rows/individuals removed):**
- `TYPAD_C = Secondaire` rows are out of scope (secondary residences); individuals
  whose *entire* history is secondary leave the corpus, the others keep their
  principal-residence rows;
- individuals whose file was administratively wiped (`depart_status =
  dossier_supprime` from notebook 3) are excluded entirely.

Deaths stay in (their episodes must exist and close with `fin_type = deces`);
they are only excluded from the *modelling* population downstream. The ~40k
snapshot-only individuals (2014 residents with zero events) are not in `events`
at all — they will enter at the episode-0 seeding step (§3).

**Spatial vocabulary — each row gets a pre- and post-state:**
- `logement` — a real dwelling `(EGID, EWID)` in a `Prive`/`Collectif` household
  (the only state where episodes exist; `EWID = 0` = dwelling not yet attributed,
  kept as a distinct key and counted);
- `administratif` — pseudo-address (`TYPE_MENAGE = Administratif` or
  `TYPADRES_C = Administrative`), not an episode location;
- `absence` — out-of-commune / detention / hospital, not an episode location;
- `indetermine` — no EGID and no recognisable pseudo-state.

**Transition detection is state-based, not family-based**: a row is a spatial
transition whenever its pre- and post-locations are both known dwellings and
differ — regardless of `CODE_MUTATION` (corrections and household events move
people too). The crosstab printed at the end quantifies exactly that.

In [3]:
# ---- Scope filters ----------------------------------------------------------
n0_rows, n0_ind = len(events), events["id_projet"].nunique()

sec_rows   = events["TYPAD_C"].eq("Secondaire")
all_sec    = events.groupby("id_projet")["TYPAD_C"].agg(lambda s: s.eq("Secondaire").all())
ind_all_sec = set(all_sec.index[all_sec])

wiped = set(target.loc[target["depart_status"].eq("dossier_supprime"), "id_projet"])

ev = events.loc[~sec_rows & ~events["id_projet"].isin(wiped)].copy()

print("Scope filters")
print(f"  input                     : {n0_rows:,} rows | {n0_ind:,} individuals")
print(f"  secondary-residence rows  : -{int(sec_rows.sum()):,} "
      f"(individuals entirely secondary: {len(ind_all_sec):,})")
print(f"  dossier_supprime          : -{len(wiped):,} individuals "
      f"({int(events['id_projet'].isin(wiped).sum()):,} rows)")
print(f"  kept                      : {len(ev):,} rows | {ev['id_projet'].nunique():,} individuals")

# ---- Spatial vocabulary -----------------------------------------------------
def norm_id(s):
    return pd.to_numeric(s, errors="coerce").astype("Int64").astype("string")

ev["egid_post"], ev["ewid_post"] = norm_id(ev["EGID"]),          norm_id(ev["MENAGE_EWID"])
ev["egid_pre"],  ev["ewid_pre"]  = norm_id(ev["MUTATION_EGID"]), norm_id(ev["MUTATION_EWID"])

def loc_state(type_menage, typadres, egid):
    out = pd.Series("indetermine", index=egid.index, dtype="string")
    out[egid.notna()] = "logement"
    out[type_menage.eq("Administratif") | typadres.eq("Administrative")] = "administratif"
    out[type_menage.eq("HorsCommune")
        | typadres.isin(["HorsCommune", "Detention", "Hospitaliere"])] = "absence"
    return out

ev["etat_post"] = loc_state(ev["TYPE_MENAGE"],          ev["TYPADRES_C"],          ev["egid_post"])
ev["etat_pre"]  = loc_state(ev["MUTATION_TYPEMENAGE"],  ev["MUTATION_TYPADRES_C"], ev["egid_pre"])

ev["loc_post"] = (ev["egid_post"] + "-" + ev["ewid_post"].fillna("?")).where(ev["etat_post"].eq("logement"))
ev["loc_pre"]  = (ev["egid_pre"]  + "-" + ev["ewid_pre"].fillna("?")).where(ev["etat_pre"].eq("logement"))

both = ev["loc_post"].notna() & ev["loc_pre"].notna()
ev["transition_spatiale"] = both & (ev["loc_post"] != ev["loc_pre"])

print("\nLocation-state distribution (post | pre)")
print(pd.concat([ev["etat_post"].value_counts(), ev["etat_pre"].value_counts()],
                axis=1, keys=["post", "pre"]).fillna(0).astype(int).to_string())

ewid0 = ev.loc[ev["etat_post"].eq("logement"), "ewid_post"].eq("0").sum()
print(f"\nEWID=0 (dwelling not yet attributed) among post 'logement': {int(ewid0):,}")
print(f"Distinct dwellings seen (post): {ev['loc_post'].nunique():,}")

print("\nSpatial transitions by event family (state-based detection)")
tt = (ev.groupby("famille_evenement")["transition_spatiale"]
        .agg(rows="size", transitions="sum"))
tt["share"] = (tt["transitions"] / tt["rows"] * 100).round(1)
print(tt.sort_values("transitions", ascending=False).to_string())

Scope filters
  input                     : 863,577 rows | 291,763 individuals
  secondary-residence rows  : -62,833 (individuals entirely secondary: 16,564)
  dossier_supprime          : -1,407 individuals (2,937 rows)
  kept                      : 797,991 rows | 273,873 individuals

Location-state distribution (post | pre)
                 post     pre
logement       786112  640593
administratif    9339   11039
absence          2535    2499
indetermine         5  143860

EWID=0 (dwelling not yet attributed) among post 'logement': 25,042
Distinct dwellings seen (post): 72,987

Spatial transitions by event family (state-based detection)
                       rows  transitions  share
famille_evenement                              
internal_move        138828       133973   96.5
arrival              176177        37396   21.2
correction            79806        36740   46.0
household             57158        14712   25.7
suppression           22384         5660   25.3
departure          

## 2. Administrative consolidation — suppressions & re-entries

Monthly files are journals of *entries*: the same real-world move can appear as an
original entry, a cancellation (`Suppression*`), and a corrected re-entry (observed
live in the Primerose → Entre-Bois → Champrilly case). Chaining must see **one
consolidated event per real move**. Two passes here (corrections-as-amendments are
handled at chaining time, §3); nothing is deleted — rows are *flagged*
(`annule`, `remplace`) and every decision is counted.

**(a) Suppression matching.** Each `Suppression*` cancels one prior `Mutation*` of
the same individual, processed in administrative order (`MUTATION_DATE`). Match
preference, per candidate stack of the target code:

1. **`lock`** — crossed-address lock: `supp.loc_pre == cand.loc_post` **and**
   `supp.loc_post == cand.loc_pre` (the suppression's `MUTATION_*` carries the
   cancelled state, its normal columns the restored one). Decisive for moves.
2. **`date`** — same `DATE_EFFECTIVE` (weak signal: on suppressions it sometimes
   carries the cancelled event's date, often the restored stay's `DATARR`).
3. **`lifo`** — most recent non-cancelled candidate (fallback, counted).
4. **`orphan`** — no candidate: the cancelled event predates the 2015
   window. **`SuppressionArriveeProvisoire` (9,653 rows) is
   structurally orphan**: no `MutationArriveeProvisoire` code exists in the
   journal — the provisional arrival it cancels is a registration action, not a
   mutation. Expected, not an anomaly; the sticky `CODETA_C` already carries the
   person-level consequence (notebook 3).

`SuppressionDossier` rows are inert here (their individuals left at §1).

**(b) Last-write-wins deduplication.** Re-entered events share
(`id_projet`, `CODE_MUTATION`, `DATE_EFFECTIVE`, destination `EGID`) — the key
deliberately excludes EWID, household and origin, because those are precisely what
re-entries correct. The administratively latest write wins; superseded rows get
`remplace = True`.

In [4]:
from collections import defaultdict, Counter

ev["annule"]        = False
ev["remplace"]      = False
ev["match_quality"] = pd.Series(pd.NA, index=ev.index, dtype="string")

is_supp   = ev["famille_evenement"].eq("suppression")   # NB3 vocabulary: famille="suppression", nature="annulation"
matchable = is_supp & ev["CODE_MUTATION"].ne("SuppressionDossier")

# ---- (a) Suppression matching, per individual in administrative order -------
ids = ev.loc[matchable, "id_projet"].unique()
sub = (ev.loc[ev["id_projet"].isin(ids),
              ["id_projet", "CODE_MUTATION", "MUTATION_DATE", "DATE_EFFECTIVE",
               "loc_pre", "loc_post"]]
         .sort_values(["id_projet", "MUTATION_DATE", "DATE_EFFECTIVE"]))

loc_pre_d, loc_post_d = sub["loc_pre"].to_dict(), sub["loc_post"].to_dict()
de_d = sub["DATE_EFFECTIVE"].to_dict()

annule_idx, quality = set(), {}
cur_pid, stacks = None, None
for t in sub.itertuples():
    if t.id_projet != cur_pid:
        cur_pid, stacks = t.id_projet, defaultdict(list)
    code = t.CODE_MUTATION
    if code.startswith("Suppression"):
        if code == "SuppressionDossier":
            continue
        target_code = code.replace("Suppression", "Mutation", 1)
        cands = [i for i in stacks[target_code] if i not in annule_idx]
        chosen, q = None, "orphan"
        if cands:
            locked = [i for i in cands
                      if pd.notna(t.loc_pre) and t.loc_pre == loc_post_d[i]
                      and pd.notna(t.loc_post) and t.loc_post == loc_pre_d[i]]
            dated  = [i for i in cands if de_d[i] == t.DATE_EFFECTIVE]
            if locked:  chosen, q = locked[-1], "lock"
            elif dated: chosen, q = dated[-1], "date"
            else:       chosen, q = cands[-1], "lifo"
        if chosen is not None:
            annule_idx.add(chosen)
            annule_idx.add(t.Index)
        quality[t.Index] = q
    elif code.startswith("Mutation"):
        stacks[code].append(t.Index)

ev.loc[list(annule_idx), "annule"] = True
qs = pd.Series(quality, dtype="string")
ev.loc[qs.index, "match_quality"] = qs

print("Suppression matching -- quality by code")
qtab = (ev.loc[matchable].groupby(["CODE_MUTATION", "match_quality"], observed=True)
          .size().unstack(fill_value=0))
qtab["total"] = qtab.sum(axis=1)
print(qtab.sort_values("total", ascending=False).to_string())
tot = qtab.drop(columns="total").sum()
print(f"\n  matched: {int(tot.get('lock',0)+tot.get('date',0)+tot.get('lifo',0)):,} "
      f"(lock {int(tot.get('lock',0)):,} | date {int(tot.get('date',0)):,} | "
      f"lifo {int(tot.get('lifo',0)):,})  orphans: {int(tot.get('orphan',0)):,}")
print(f"  rows neutralised (pairs): {len(annule_idx):,}")

# ---- (b) Last-write-wins deduplication of re-entries -------------------------
active = ~ev["annule"] & ~is_supp
tmp = (ev.loc[active, ["id_projet", "CODE_MUTATION", "DATE_EFFECTIVE", "MUTATION_DATE"]]
         .assign(egid=ev.loc[active, "egid_post"].fillna("~none~"))
         .sort_values("MUTATION_DATE"))
dup = tmp.duplicated(subset=["id_projet", "CODE_MUTATION", "DATE_EFFECTIVE", "egid"],
                     keep="last")
ev.loc[dup.index[dup], "remplace"] = True

print("\nRe-entry deduplication (last write wins) -- superseded rows by family")
print(ev.loc[ev["remplace"], "famille_evenement"].value_counts().to_string())

n_out = int((~ev["annule"] & ~ev["remplace"] & ~is_supp).sum())
print(f"\nConsolidated event pool: {n_out:,} rows "
      f"(from {len(ev):,}: -{int(ev['annule'].sum()):,} annule, "
      f"-{int(ev['remplace'].sum()):,} remplace, "
      f"-{int((is_supp & ~ev['annule']).sum()):,} unmatched suppression rows)")

Suppression matching -- quality by code
match_quality                       date  lifo  lock  orphan  total
CODE_MUTATION                                                      
SuppressionArriveeProvisoire           0     0     0    8294   8294
SuppressionDepartDefinitif            18  2097  3857     228   6200
SuppressionDemenagement              423   700  1474      64   2661
SuppressionDepartNonConfirme           6   665  1376       1   2048
SuppressionDepartAnticipe              3   116  1432       7   1558
SuppressionArriveeDefinitive         212    32   116     470    830
SuppressionAdresseAdministrative      13   568     2       2    585
SuppressionDemenagementHorsCommune     4   118     1       3    126
SuppressionDeces                       0     8    44       2     54
SuppressionDetention                   1    13     0       1     15
SuppressionHospitalisation             2    10     0       1     13

  matched: 13,311 (lock 8,302 | date 682 | lifo 4,327)  orphans: 9,073
  ro

## 3. Episode 0 seeding & sequential chaining

One pass per individual over the consolidated pool, in **biographical order**
(`DATE_EFFECTIVE`, tie-broken by `MUTATION_DATE`). Events effective *after* the
censor date are deferred (counted): the observation window ends 2026-05-31.

**Roles.** Each consolidated event plays one role: `ouverture` (arrival, birth),
`transition` (state-based spatial move — any family), `fermeture` (definitive
departure; non-confirmed departure closes immediately; announced departure closes
at `DATDEP` once reached, else flags the episode `depart_annonce`), `deces`,
`absence` (out-of-commune / hospital / detention / administrative pseudo-address:
the episode stays open and flagged, an interval goes to the `absences` side
table), or `attribut` (inert for chaining).

**Seeding.** Every 2014 resident (non-wiped, non-secondary, dwelling state) opens
episode 0 at the snapshot dwelling. `date_debut = DATARR` — a **left bound**: it
dates the arrival in the commune, not necessarily in that dwelling (pre-2015
internal moves are unobservable), flagged via `debut_type = snapshot`. Residents
never seen in events yield a single censored episode.

**Continuity is checked, never enforced.** At each transition, the previous
episode's dwelling is compared to the event's pre-state (`loc_pre`);
`continuite_entree_ok = False` marks explained-or-not discontinuities (suppression
bridges, snapshot divergence — quantified in §4). An arrival
while an episode is open closes it as `pont_rearrivee` (unobserved departure).
A dwelling-state event with no open episode lazily opens one
(`debut_type = reprise`, counted).

Right-censoring: episodes still open at the end get `fin_type = censure`
(`censure_depart_annonce` if a pending announced departure flags them).

**Refinements from the first full run.** (1) The register's real departure flow is
*announce → definitive confirmation*: the definitive event lands after the episode
was already closed at `DATDEP`. Closures arriving after a departure-type closure
are therefore counted as `confirmation_depart` (resp. `re_annonce_apres_fermeture`),
not as anomalies; the anomaly counter `fermeture_sans_episode` keeps only true
unseen-start endings (filtered openings). (2) **EWID attribution**:
a transition to the same EGID whose outgoing EWID is `0`/unknown is the register
refining the dwelling id retroactively — the open episode is amended in place
(`attribution_ewid`), not split.

**(3) Same-dwelling amendments.** A state-based transition whose destination equals
the *open episode's* current dwelling is not a move: the row's `MUTATION_*`
pre-state encodes something non-residential (household recomposition, admin
correction) while the dwelling is unchanged. Detected at row level, these are
false transitions; the open episode is amended (`amend_meme_logement`), never
split. This removes the phantom episodes that dominated the first run's
unexplained ruptures.

In [5]:
# ---- Consolidated chain pool, roles, censoring -------------------------------
chain = ev.loc[~ev["annule"] & ~ev["remplace"]
               & ~ev["famille_evenement"].eq("suppression")].copy()

n_future = int((chain["DATE_EFFECTIVE"] > CENSOR_DATE).sum())
chain = chain.loc[chain["DATE_EFFECTIVE"] <= CENSOR_DATE]

fam = chain["famille_evenement"]
chain["role"] = "attribut"
chain.loc[chain["transition_spatiale"], "role"] = "transition"
chain.loc[fam.isin(["arrival", "birth"]), "role"] = "ouverture"
chain.loc[fam.eq("departure"), "role"] = "fermeture_depart"
chain.loc[chain["CODE_MUTATION"].eq("MutationDepartNonConfirme"), "role"] = "fermeture_depart_nc"
chain.loc[chain["CODE_MUTATION"].eq("MutationDepartAnticipe"),   "role"] = "annonce_depart"
chain.loc[fam.eq("death"), "role"] = "fermeture_deces"
abs_like = chain["etat_post"].isin(["absence", "administratif"]) & chain["role"].isin(["attribut"])
chain.loc[fam.eq("temporary_absence") | abs_like, "role"] = "absence"

print(f"Chain pool: {len(chain):,} rows | deferred (effective after censor): {n_future:,}")
print(chain["role"].value_counts().to_string())

# ---- Seeds from the 2014 snapshot --------------------------------------------
def col(df, name):
    return df[name] if name in df.columns else pd.Series(pd.NA, index=df.index)

snap = pop2014.copy()
n_snap0 = len(snap)
if "TYPAD_C" in snap.columns:
    snap = snap.loc[~snap["TYPAD_C"].eq("Secondaire")]
snap = snap.loc[~snap["id_projet"].isin(wiped)]
snap["egid_n"] = norm_id(col(snap, "EGID"))
snap["ewid_n"] = norm_id(col(snap, "MENAGE_EWID"))
snap["etat0"]  = loc_state(col(snap, "TYPE_MENAGE").astype("string"),
                           col(snap, "TYPADRES_C").astype("string"), snap["egid_n"])
snap = snap.loc[snap["etat0"].eq("logement")]
snap["loc0"] = snap["egid_n"] + "-" + snap["ewid_n"].fillna("?")
BORNE_2014 = pd.Timestamp("2014-12-31")
snap["date0"] = col(snap, "DATARR").fillna(BORNE_2014)
seeds = snap.set_index("id_projet")[["loc0", "date0"]]
seeds["menage0"] = col(snap.set_index("id_projet"), "NUMERO_MENAGE")
print(f"\nSeeds: {len(seeds):,} 2014 residents in a dwelling "
      f"(snapshot: {n_snap0:,}; excluded: secondary/wiped/non-dwelling {n_snap0 - len(seeds):,})")

# ---- Sequential chaining ------------------------------------------------------
from collections import Counter
FIN_TRANSITION = {"internal_move": "demenagement", "correction": "correction",
                  "household": "recomposition_menage"}
ABS_TYPE = {"MutationDemenagementHorsCommune": "hors_commune",
            "MutationDetention": "detention", "MutationHospitalisation": "hopital"}

episodes, absences, stats = [], [], Counter()

def new_ep(pid, loc, date, debut_type, menage, cont=True):
    return {"id_projet": pid, "loc": loc, "date_debut": date, "debut_type": debut_type,
            "menage_debut": menage, "absence": False, "depart_annonce": False,
            "continuite_entree_ok": cont, "loc_pre_evt": pd.NA,
            "date_fin": pd.NaT, "fin_type": pd.NA}

def close_ep(ep, date, fin_type):
    ep["date_fin"], ep["fin_type"] = date, fin_type
    episodes.append(ep)

chain_sorted = chain.sort_values(["id_projet", "DATE_EFFECTIVE", "MUTATION_DATE"])
seen_ids = set()
n_total = chain_sorted["id_projet"].nunique()

for i, (pid, grp) in enumerate(chain_sorted.groupby("id_projet", sort=False), 1):
    if i % 25_000 == 0:
        print(f"  chaining: {i:,}/{n_total:,} individuals", flush=True)
    seen_ids.add(pid)
    open_ep, abs_open, last_close = None, None, None
    if pid in seeds.index:
        s = seeds.loc[pid]
        open_ep = new_ep(pid, s["loc0"], s["date0"], "snapshot", s["menage0"])
    for t in grp.itertuples():
        date, role = t.DATE_EFFECTIVE, t.role
        if abs_open is not None and t.etat_post == "logement" and role != "absence":
            abs_open["date_fin"] = date; absences.append(abs_open); abs_open = None
        if role == "ouverture":
            if t.etat_post == "logement":
                last_close = None          # a new stay begins: reset departure memory
                if open_ep is not None:
                    close_ep(open_ep, date, "pont_rearrivee"); stats["pont_rearrivee"] += 1
                open_ep = new_ep(pid, t.loc_post, date,
                                 "naissance" if t.famille_evenement == "birth" else "arrivee",
                                 t.NUMERO_MENAGE)
            else:
                stats["ouverture_hors_logement"] += 1
                if abs_open is None:
                    abs_open = {"id_projet": pid, "type": t.etat_post,
                                "date_debut": date, "date_fin": pd.NaT}
        elif role == "transition":
            if open_ep is not None and pd.notna(t.loc_post):
                if t.loc_post == open_ep["loc"]:
                    # Same dwelling reached: the row's MUTATION_* encodes a
                    # non-residential pre-state (household recomposition, admin
                    # correction). No move -- amend attributes, do not split.
                    open_ep["menage_debut"] = t.NUMERO_MENAGE
                    stats["amend_meme_logement"] += 1
                    continue
                e0, _, w0 = str(open_ep["loc"]).partition("-")
                e1, _, w1 = str(t.loc_post).partition("-")
                if e0 == e1 and w0 in ("0", "?") and w1 not in ("0", "?"):
                    # EWID attribution: the register refines the dwelling id
                    # retroactively -- amend the open episode, do not split it.
                    open_ep["loc"] = t.loc_post
                    stats["attribution_ewid"] += 1
                    continue
            cont = open_ep is not None and pd.notna(t.loc_pre) and open_ep["loc"] == t.loc_pre
            if open_ep is not None:
                close_ep(open_ep, date, FIN_TRANSITION.get(t.famille_evenement, "transition"))
                open_ep = new_ep(pid, t.loc_post, date, "transition", t.NUMERO_MENAGE, cont)
                open_ep["loc_pre_evt"] = t.loc_pre
                stats["continuite_ok" if cont else "continuite_rompue"] += 1
            else:
                open_ep = new_ep(pid, t.loc_post, date, "reprise", t.NUMERO_MENAGE, False)
                stats["transition_sans_episode"] += 1
        elif role in ("fermeture_depart", "fermeture_depart_nc"):
            if open_ep is not None:
                last_close = "depart" if role == "fermeture_depart" else "depart_non_confirme"
                close_ep(open_ep, date, last_close)
                open_ep = None
            elif last_close in ("depart_anticipe", "depart_non_confirme", "depart"):
                stats["confirmation_depart"] += 1   # announce -> definitive pattern
            else:
                stats["fermeture_sans_episode"] += 1
        elif role == "annonce_depart":
            if pd.notna(t.DATDEP) and t.DATDEP <= CENSOR_DATE:
                if open_ep is not None:
                    close_ep(open_ep, t.DATDEP, "depart_anticipe"); open_ep = None
                    last_close = "depart_anticipe"
                elif last_close in ("depart_anticipe", "depart_non_confirme", "depart"):
                    stats["re_annonce_apres_fermeture"] += 1
                else:
                    stats["fermeture_sans_episode"] += 1
            elif open_ep is not None:
                open_ep["depart_annonce"] = True; stats["depart_annonce_pendant"] += 1
        elif role == "fermeture_deces":
            d = t.DDECES if pd.notna(t.DDECES) else date
            if open_ep is not None:
                close_ep(open_ep, d, "deces"); open_ep = None
                last_close = "deces"
            else:
                stats["deces_sans_episode"] += 1
        elif role == "absence":
            if open_ep is not None:
                open_ep["absence"] = True
            if abs_open is None:
                abs_open = {"id_projet": pid,
                            "type": ABS_TYPE.get(t.CODE_MUTATION, t.etat_post),
                            "date_debut": date, "date_fin": pd.NaT}
        else:  # attribut
            if open_ep is None and t.etat_post == "logement" and pd.notna(t.loc_post):
                open_ep = new_ep(pid, t.loc_post, date, "reprise", t.NUMERO_MENAGE, False)
                stats["reprise_attribut"] += 1
    if abs_open is not None:
        absences.append(abs_open)
    if open_ep is not None:
        close_ep(open_ep, pd.NaT,
                 "censure_depart_annonce" if open_ep["depart_annonce"] else "censure")

# Snapshot-only individuals (never seen in events): one censored episode 0
for pid, s in seeds.loc[~seeds.index.isin(seen_ids)].iterrows():
    ep = new_ep(pid, s["loc0"], s["date0"], "snapshot", s["menage0"])
    close_ep(ep, pd.NaT, "censure")
    stats["seed_only"] += 1

episodes = pd.DataFrame(episodes)
absences = pd.DataFrame(absences)
episodes["ep_num"] = episodes.groupby("id_projet").cumcount()

print(f"\nEpisodes: {len(episodes):,} | individuals: {episodes['id_projet'].nunique():,} "
      f"| absences intervals: {len(absences):,}")
print("\ndebut_type:"); print(episodes["debut_type"].value_counts().to_string())
print("\nfin_type:");   print(episodes["fin_type"].value_counts().to_string())
print("\nChaining stats:")
for k, v in sorted(stats.items()):
    print(f"  {k:28} {v:,}")
n_trans = stats["continuite_ok"] + stats["continuite_rompue"]
if n_trans:
    print(f"\nContinuity at transitions: {stats['continuite_ok']/n_trans*100:.1f}% OK "
          f"({stats['continuite_rompue']:,} broken -- dissected in §4)")
print("\nEpisodes per individual:")
print(episodes.groupby("id_projet").size().describe().round(2).to_string())

Chain pool: 751,143 rows | deferred (effective after censor): 125
role
ouverture              194199
transition             178959
fermeture_depart       172832
attribut                78993
annonce_depart          68196
fermeture_depart_nc     41524
fermeture_deces         11600
absence                  4840

Seeds: 140,003 2014 residents in a dwelling (snapshot: 140,228; excluded: secondary/wiped/non-dwelling 225)
  chaining: 25,000/268,973 individuals
  chaining: 50,000/268,973 individuals
  chaining: 75,000/268,973 individuals
  chaining: 100,000/268,973 individuals
  chaining: 125,000/268,973 individuals
  chaining: 150,000/268,973 individuals
  chaining: 175,000/268,973 individuals
  chaining: 200,000/268,973 individuals
  chaining: 225,000/268,973 individuals
  chaining: 250,000/268,973 individuals

Episodes: 493,912 | individuals: 310,746 | absences intervals: 4,496

debut_type:
debut_type
arrivee       175176
snapshot      140003
transition    133408
reprise        26902
naiss

## 4. Validation, rupture dissection, statistics & export

Four checks, then the exports.

**(a) Structural integrity** — episodes of one individual must not overlap and must
have non-negative durations (retroactive corrections effective *before* the
episode's left bound can produce negatives; counted, not hidden).

**(b) Rupture dissection.** The 23% of transitions whose entry pre-state does not
match the previous episode's dwelling are classified by *known cause*, in priority
order: `premiere_apres_snapshot` (episode 0's dwelling diverges from the first
observed move — the 4–7% disagreement measured in notebook 4),
`pont_suppression` (a cancelled event sits between the two episodes),
`renum_ewid_meme_egid` (same building, different dwelling id — register
renumbering), `apres_reprise` (the previous episode itself started blind).
The remainder — `inexplique` — is the table's final uncertainty figure.
Corpus completeness is verified upstream (notebooks 1-2: every month present
and non-empty), so no rupture cause is attributable to observation gaps.

**(c) Reconciliation with the notebook-3 target** — each individual's *last*
episode ending is cross-tabulated against `depart_status`: departure-type endings
should meet `departed_*`, `deces` should meet `death`, `censure` should meet
`present`/`pending`. Snapshot-only individuals sit outside `person_target`
(by construction) and are reported separately — they enter the modelling
population as certain `present`.

**(d) Descriptive statistics** — durations (censored at 2026-05-31), share
censored, dwellings per trajectory: the first EDA numbers of the project.

**Exports**: `episodes.parquet`, `absences.parquet`, and `pipeline_synthese.csv` —
every count produced across §1–§4, in one table, so the report's *Data* section
is a copy-paste rather than a hunt through cell outputs.

In [6]:
###### ---- (a) Structural integrity -------------------------------------------------
eps = episodes.sort_values(["id_projet", "date_debut", "ep_num"]).copy()
eps["ep_num"] = eps.groupby("id_projet").cumcount()
eps["duree_j"] = (eps["date_fin"].fillna(CENSOR_DATE) - eps["date_debut"]).dt.days

neg = int((eps["duree_j"] < 0).sum())
prev_fin = eps.groupby("id_projet")["date_fin"].shift()
eps["overlap"] = eps["date_debut"] < prev_fin
overlap = int(eps["overlap"].sum())
print(f"(a) Integrity -- negative durations: {neg:,} | overlaps: {overlap:,} of {len(eps):,}")
if overlap or neg:
    # retro-active corrections effective before the running episode's start:
    # clip the start to the previous end so episodes stay non-overlapping, flag it.
    eps["debut_ajuste"] = eps["overlap"] & prev_fin.notna()
    eps.loc[eps["debut_ajuste"], "date_debut"] = prev_fin[eps["debut_ajuste"]]
    eps["duree_j"] = (eps["date_fin"].fillna(CENSOR_DATE) - eps["date_debut"]).dt.days
    # Intra-episode inversion: a retro-active closure effective before the episode's
    # own start -> zero-length episode, flagged (kept, not dropped).
    inv = eps["duree_j"] < 0
    eps["duree_inversee"] = inv
    eps.loc[inv, "date_fin"] = eps.loc[inv, "date_debut"]
    eps["duree_j"] = (eps["date_fin"].fillna(CENSOR_DATE) - eps["date_debut"]).dt.days
    print(f"    -> {int(eps['debut_ajuste'].sum()):,} starts clipped to previous end; "
          f"{int(inv.sum()):,} intra-episode inversions zeroed "
          f"(retro-active closures); residual negatives: {int((eps['duree_j']<0).sum()):,}")

# ---- (b) Rupture dissection ----------------------------------------------------
prev = eps.groupby("id_projet").shift()
mask_rupt = eps["debut_type"].eq("transition") & ~eps["continuite_entree_ok"].astype(bool)
rupt = eps.loc[mask_rupt, ["id_projet", "loc", "loc_pre_evt", "date_debut"]].copy()
rupt["prev_debut_type"] = prev.loc[rupt.index, "debut_type"]
rupt["prev_loc"]        = prev.loc[rupt.index, "loc"].astype("string")
rupt["prev_debut"]      = prev.loc[rupt.index, "date_debut"]

def egid_of(s):
    return s.astype("string").str.partition("-")[0]

# cancelled event inside the unobserved window?
ann = (ev.loc[ev["annule"] & ~ev["famille_evenement"].eq("suppression"),
              ["id_projet", "DATE_EFFECTIVE"]].rename(columns={"DATE_EFFECTIVE": "d_ann"}))
m = rupt.reset_index().merge(ann, on="id_projet", how="left")
m["in_win"] = (m["d_ann"] > m["prev_debut"]) & (m["d_ann"] <= m["date_debut"])
has_supp = m.groupby("index")["in_win"].any()
rupt["pont_supp"] = has_supp.reindex(rupt.index, fill_value=False)

def _b(s):
    return pd.Series(s, index=rupt.index).astype("boolean").to_numpy(dtype=bool, na_value=False)

cond = [
    _b(rupt["prev_debut_type"].eq("snapshot")),
    _b(rupt["pont_supp"]),
    _b(egid_of(rupt["prev_loc"]).eq(egid_of(rupt["loc_pre_evt"]))),
    _b(rupt["prev_debut_type"].isin(["reprise"])),
]
lab = ["premiere_apres_snapshot", "pont_suppression",
       "renum_ewid_meme_egid", "apres_reprise"]
rupt["cause"] = np.select(cond, lab, default="inexplique")

n_trans = int(eps["debut_type"].eq("transition").sum())
print(f"\n(b) Ruptures: {len(rupt):,} of {n_trans:,} transitions")
tab = rupt["cause"].value_counts()
for k, v in tab.items():
    print(f"    {k:26} {v:>7,}  ({v/n_trans*100:.1f}% of transitions)")
print(f"    -> unexplained residual: {int(tab.get('inexplique',0)):,} "
      f"({tab.get('inexplique',0)/n_trans*100:.1f}% of transitions) -- final uncertainty figure")

# ---- (c) Reconciliation with person_target -------------------------------------
# Reload so this section can be re-run standalone.
person_target = load("person_target")
last = eps.groupby("id_projet").tail(1).set_index("id_projet")
tgt  = person_target.set_index("id_projet")["depart_status"]
rec  = last[["fin_type"]].join(tgt, how="left")
rec["depart_status"] = rec["depart_status"].fillna("(hors_target: snapshot-only)")
print("\n(c) Last-episode ending x notebook-3 target")
print(pd.crosstab(rec["fin_type"], rec["depart_status"]).to_string())

dep_fin = rec["fin_type"].isin(["depart", "depart_anticipe", "depart_non_confirme"])
in_tgt  = ~rec["depart_status"].str.startswith("(hors")
dep_tgt = rec["depart_status"].str.startswith("departed")
agree = (dep_fin.eq(dep_tgt) & in_tgt).sum() / in_tgt.sum()
print(f"\n    departure agreement (episodes vs target), on shared individuals: {agree*100:.1f}%")

# ---- (d) Descriptive statistics -------------------------------------------------
closed = eps["date_fin"].notna()
print(f"\n(d) Episodes: {len(eps):,} | censored: {int((~closed).sum()):,} "
      f"({(~closed).mean()*100:.1f}%)")
print("    duration (days), closed episodes:")
print(eps.loc[closed, "duree_j"].describe().round(1).to_string())
print("    duration (days), censored (to 2026-05-31):")
print(eps.loc[~closed, "duree_j"].describe().round(1).to_string())
print("\n    absence intervals by type:")
print(absences["type"].value_counts().to_string() if len(absences) else "    (none)")

# ---- Export + pipeline synthesis ------------------------------------------------
def save(df, stem):
    pq = DATA_DIR / f"{stem}.parquet"
    try:
        df.to_parquet(pq, index=False); return pq
    except Exception as e:
        gz = pq.with_suffix(".csv.gz")
        df.to_csv(gz, index=False, compression="gzip")
        print(f"  {stem}: parquet unavailable ({type(e).__name__}) -> CSV gzip"); return gz

qs_tab = ev.loc[ev["famille_evenement"].eq("suppression"), "match_quality"].value_counts()
synth = pd.DataFrame([
    ("§1 input",           "rows / individuals",           f"{n0_rows:,} / {n0_ind:,}"),
    ("§1 scope",           "secondary rows removed",       f"{int(sec_rows.sum()):,}"),
    ("§1 scope",           "entirely-secondary individuals", f"{len(ind_all_sec):,}"),
    ("§1 scope",           "dossier_supprime individuals", f"{len(wiped):,}"),
    ("§2 suppressions",    "matched lock / date / lifo",   f"{int(qs_tab.get('lock',0)):,} / {int(qs_tab.get('date',0)):,} / {int(qs_tab.get('lifo',0)):,}"),
    ("§2 suppressions",    "orphans (incl. structural)",   f"{int(qs_tab.get('orphan',0)):,}"),
    ("§2 dedup",           "superseded re-entries",        f"{int(ev['remplace'].sum()):,}"),
    ("§3 chaining",        "deferred post-censor events",  f"{n_future:,}"),
    ("§3 chaining",        "seeds / seed-only",            f"{len(seeds):,} / {stats['seed_only']:,}"),
    ("§3 chaining",        "departure confirmations",      f"{stats['confirmation_depart']:,}"),
    ("§3 chaining",        "EWID attributions amended",    f"{stats['attribution_ewid']:,}"),
    ("§3 chaining",        "true unseen-start endings",    f"{stats['fermeture_sans_episode']:,}"),
    ("§4 episodes",        "episodes / individuals",       f"{len(eps):,} / {eps['id_projet'].nunique():,}"),
    ("§4 episodes",        "censored share",               f"{(~closed).mean()*100:.1f}%"),
    ("§4 ruptures",        "unexplained residual",         f"{int(tab.get('inexplique',0)):,} ({tab.get('inexplique',0)/n_trans*100:.1f}% of transitions)"),
    ("§4 reconciliation",  "departure agreement vs target", f"{agree*100:.1f}%"),
], columns=["etape", "indicateur", "valeur"])

p1 = save(eps.drop(columns=["depart_annonce"]), "episodes")
p2 = save(absences, "absences")
p3 = save(synth, "pipeline_synthese")
print(f"\nSaved: {p1.name} | {p2.name} | {p3.name}")
print("\n=== Pipeline synthesis (report-ready) ===")
print(synth.to_string(index=False))

(a) Integrity -- negative durations: 4 | overlaps: 975 of 493,912
    -> 975 starts clipped to previous end; 358 intra-episode inversions zeroed (retro-active closures); residual negatives: 0

(b) Ruptures: 4,723 of 133,408 transitions
    premiere_apres_snapshot      2,594  (1.9% of transitions)
    inexplique                   1,308  (1.0% of transitions)
    pont_suppression               434  (0.3% of transitions)
    renum_ewid_meme_egid           299  (0.2% of transitions)
    apres_reprise                   88  (0.1% of transitions)
    -> unexplained residual: 1,308 (1.0% of transitions) -- final uncertainty figure

(c) Last-episode ending x notebook-3 target
depart_status           (hors_target: snapshot-only)  death  departed_anticipated_past  departed_confirmed  departed_nonconfirmed  pending_anticipated  present
fin_type                                                                                                                                                       
cens